# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant-compliant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"[1m{metadata.name}[0m: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs referenced by their `@id`. These can be discovered from the Croissant metadata.

In [ ]:
# List all record sets with their @id and field summary
print('\nAvailable record sets:')
for recset in metadata.record_sets:
    print(f"- @id: {recset.id}")
    print(f"    Name: {recset.name}")
    print(f"    Fields:")
    for field in recset.fields:
        print(f"      - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare DataFrames for each record set
record_sets_ids = [r.id for r in metadata.record_sets]
dataframes = {}
for recset_id in record_sets_ids:
    records = list(dataset.records(record_set=recset_id))
    if records:
        dataframes[recset_id] = pd.DataFrame(records)
    else:
        dataframes[recset_id] = pd.DataFrame()

# Show columns of the first record set
if len(record_sets_ids) > 0:
    first_recset = record_sets_ids[0]
    print(f"Columns in {first_recset}:")
    print(dataframes[first_recset].columns.tolist())
    display(dataframes[first_recset].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Make sure to reference fields by their `@id`.

In [ ]:
# For example, select a numeric field for analysis, such as coverage or MW (Molecular Weight)

# Find a numeric field (float/integer) from first record set
import numpy as np
first_recset_obj = next((r for r in metadata.record_sets if r.id == first_recset), None)
numeric_fields = [f for f in first_recset_obj.fields if f.data_type in ['http://schema.org/Float', 'http://schema.org/Integer', 'Float', 'Integer']]
if numeric_fields:
    numeric_field = numeric_fields[0].id
    print(f"Selected numeric_field for analysis: {numeric_field}")
else:
    print("No numeric fields found.")

# Example threshold for filtering
threshold = 10
df = dataframes[first_recset]

# Check field available
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())
    # Normalization
    m = filtered_df[numeric_field].astype(float).mean()
    s = filtered_df[numeric_field].astype(float).std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - m) / s
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a string field, e.g., 'description', if available
    string_fields = [f for f in first_recset_obj.fields if f.data_type in ['http://schema.org/Text', 'Text', 'http://schema.org/Name', 'Name']]
    group_field = string_fields[0].id if string_fields else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print(f"Numeric field {numeric_field} not available in DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn as appropriate.

In [ ]:
# Basic histogram of the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in {first_recset}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load a Croissant-compliant biomedical dataset using the `mlcroissant` library, review metadata, extract tabular data, conduct basic EDA, and visualize core features. For more advanced workflows, consider exploring relationships between samples, cross-referencing by accession/description, or integrating with external protein annotation tools using the provided `@id`s.